## Code from QTrack
Used to merge modified u and v winds from era5 with modified u and v winds from wrfout files.
Slightly modified by A. Thornton

In [19]:
#Quinton Lawton, University of Miami, 2022
# ## Combine 700hPa wind files between ERA5 and MPAS, ultimate goal to create one large CV file for tracking

from datetime import datetime, timedelta

import numpy as np
import xarray as xr
import pandas as pd
import os
from AEW_module import season, AEW, AEW_CCKW

# for regridding
import xesmf as xe
import qtrack
import pickle
from qtrack.curvvort import compute_curvvort
from qtrack.tracking import run_postprocessing, run_tracking


In [20]:
### THIS CELL WAS ADDED BY A. THORNTON

## is this a restart run?
flux = False
# what is the base initialization time? (ens member)
init_time = pd.to_datetime('2020-09-03 12:00:00')
init_string = init_time.strftime('%d%H')
# what time did you turn on the fluxes?
fluxon_time = pd.to_datetime('2020-09-05 12:00:00')

In [21]:
### THIS CELL WAS ADDED BY A. THORNTON 

# this string is used to find that experiment
string = fluxon_time.strftime('%d%H')
year = init_time.strftime('%Y')
if year == '2011':
    subdir = 'long_lived_case'
    if int(init_string) % 2 == 0:
        end_time = pd.to_datetime('2011-08-29 12:00') # Even 
    else:
        end_time = pd.to_datetime('2011-08-29 09:00') # Odd 
else:
    subdir = 'cent_atl_case'
    end_time = pd.to_datetime('2020-09-09 12:00')  

# for indexing...
date_list = pd.date_range(start=init_time, end=end_time, freq='3h')
fluxon_index = np.where(date_list==fluxon_time)[0][0]
response = fluxon_index + 3 # 3 days later, to respond to fluxes on

# Set directory where wrfout files reside, and list the files for processing.  Set up for a directory with only wrfout files.
if flux == True:
    plt_name = 'rst_on'+string+'z'
    f = pd.Timedelta(init_time - fluxon_time).total_seconds() 
    hours = int((f / 3600)*-1)
    run_name = 'rst_on'+str(hours)
else:
    run_name = 'fluxoff'
    #run_name = 'fluxon'
    plt_name = run_name
    hours = run_name

os.chdir("/glade/campaign/univ/uncs0067/flux_experiments/"+subdir+"/init"+init_string+"z/"+run_name+"/")
plotsdir = '/glade/u/home/athornton/wrf_visualization/plots/restart/init'+init_string+'z/'+plt_name+'/'

title = init_string+", "+run_name+", "+string+"z"
save_name = run_name +"_"+ init_string
print(save_name)

fluxoff_0312


In [13]:
### Settings
year_in = year
cut_start = init_time.strftime('%Y-%m-%d-%H') #The start time of the MPAS data
cut_end =  end_time.strftime('%Y-%m-%d-%H') #The end of the MPAS data
cut_start_early = (init_time - timedelta(hours=6)).strftime('%Y-%m-%d-%H') #6 hours before the MPAS start -- basically, where the ERA5 data will end when merged
era5_start = (init_time - timedelta(days=10)).strftime('%Y-%m-%d-%H') #Where we want to start the ERA5 data from when merging
save_data = True # Save output of data

out_name = 'wind_for_tracking_'+save_name #What you want to call
hr_delta = 3 #Time delta of the MPAS output (hours)
hr_plot_delta = 3 #Time delta of DESIRED output (hours, 6 hours for AEW tracking is good)

### *************** WARNING: HARDWIRED PATHS, CHANGE  **************
mpas_in = '/glade/u/home/athornton/qtrack/wind_files/wrfout_regrid_'+save_name+'.nc' 
era5_in = '/glade/u/home/athornton/qtrack/wind_files/wind_season_lowres_700_'+year_in+'_B1-6hr.nc' #ERA5 data in

### *************** WHAT YOU WANT TO SAVE THE FILE AS **************
out_file = '/glade/u/home/athornton/qtrack/wind_files/'+out_name+'.nc'
### ----------------------------------------------------------------

In [14]:
### LOAD IN THE DATA
mpas_xr = xr.open_dataset(mpas_in, chunks = 'auto')
era5_xr = xr.open_dataset(era5_in, chunks = 'auto').sel(time = slice(era5_start, cut_start_early))

## Next, we want to round the coordinates and rename the variables in the MPAS files
lat_round = np.round(mpas_xr.coords['latitude'].values)
lon_round = np.round(mpas_xr.coords['longitude'].values)
#(Rounding is necessary because sometimes convert_mpas outputs lons like 19.9999999999 instead of 20)

# Replace with the real value
mpas_xr.coords['longitude'] = lon_round
mpas_xr.coords['latitude'] = lat_round

In [15]:
#For later slicing
lat_min_mpas = mpas_xr.latitude.values[0]
lat_max_mpas = mpas_xr.latitude.values[-1]

#Note that the latitudes are reversed in ERA5 hence the denotation, but double check your data
lat_min = era5_xr.latitude.values[-1]
lat_max = era5_xr.latitude.values[0]
lon_min = era5_xr.longitude.values[0]
lon_max = era5_xr.longitude.values[-1]

#Flip and cut down the era5 data
era5_xr = era5_xr.reindex(latitude=era5_xr.latitude[::-1]).sel(latitude = slice(lat_min_mpas, lat_max_mpas))

#Finally, cut down the longitude of the era5_data
mpas_xr = mpas_xr.sel(longitude = slice(lon_min, lon_max), latitude = slice(lat_min, lat_max))


##### --- UPDATING TIME FOR MPAS ----- #####
### We also want to make a list of times to represented the start and end dates contained in the files
cut_start_dt = datetime.strptime(cut_start, '%Y-%m-%d-%H')
cut_end_dt = datetime.strptime(cut_end, '%Y-%m-%d-%H')
length_dt = int((cut_end_dt - cut_start_dt).total_seconds()/3600/hr_delta)+1 #Not sure about the plus one here but it works....
timedelta_out = timedelta(hours=hr_delta)

date_list_dt = [cut_start_dt + timedelta_out*hour for hour in range(length_dt)]

date_list = [datetime.strftime(in_dt, '%Y-%m-%d-%H') for in_dt in date_list_dt]
date_list_6hr = date_list[::2]
date_list_dt_6hr = date_list_dt[::2]

### Now put it in the data and cut it down
mpas_xr.coords['time'] = date_list_dt
#mpas_xr = mpas_xr.sel(time = date_list_dt_6hr)


##### --- MERGE THE TWO ARRAYS AND SAVE --- #####
if save_data == True:
    merged = xr.concat([era5_xr, mpas_xr], dim = 'time')
    merged = merged.fillna(0)
    merged.to_netcdf(out_file)


In [16]:
out_file

'/glade/u/home/athornton/qtrack/wind_files/wind_for_tracking_fluxoff_0312.nc'